# Meteo Cluster

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from pathlib import Path

## Подготовка данных

In [2]:
from src.data import load_data, dataset_to_dataframe, normalize_dataframe
from config import config

dataset_path = config.data_dir / "2025_1000.nc"

ds = load_data(file_path=dataset_path, pressure_level=0)

In [3]:
ds

<xarray.Dataset> Size: 11MB
Dimensions:  (time: 61, lat: 91, lon: 120)
Coordinates:
  * time     (time) datetime64[ns] 488B 2025-11-01T10:30:00 ... 2025-12-31T10...
  * lat      (lat) float64 728B -90.0 -88.0 -86.0 -84.0 ... 84.0 86.0 88.0 90.0
  * lon      (lon) float64 960B 0.0 3.0 6.0 9.0 12.0 ... 348.0 351.0 354.0 357.0
    lev      float64 8B 1e+03
Data variables:
    H        (time, lat, lon) float32 3MB ...
    T        (time, lat, lon) float32 3MB ...
    U        (time, lat, lon) float32 3MB ...
    V        (time, lat, lon) float32 3MB ...
Attributes: (12/35)
    CDI:                               Climate Data Interface version 1.7.1 (...
    Conventions:                       CF-1
    history:                           Wed Jan 28 15:09:34 2026: cdo sellevel...
    _NCProperties:                     version=1|netcdflibversion=4.5.0|hdf5l...
    History:                           Original file generated: Tue Nov 11 21...
    Comment:                           GMAO filename: d5124_m2_jan10.inst3_3d...
    ...                                ...
    RangeBeginningTime:                00:00:00.000000
    RangeEndingDate:                   2025-11-01
    RangeEndingTime:                   21:00:00.000000
    frequency:                         day
    NCO:                               4.3.7
    CDO:                               Climate Data Operators version 1.7.1 (...

In [4]:
meteo_df = dataset_to_dataframe(ds, features=["H", "T", "U", "V"])

In [5]:
meteo_df.head(5)

H         T         U         V     lev
time                lon lat                                                  
2025-11-01 10:30:00 0.0 -90.0 -5.423445 -4.133301  4.570474  2.031699  1000.0
                        -88.0 -5.423445 -4.133301  4.570474  2.031699  1000.0
                        -86.0 -5.423445 -4.133301  4.570474  2.031699  1000.0
                        -84.0 -5.847328 -3.828979  4.720197  1.897010  1000.0
                        -82.0 -6.308565 -3.814880  4.802876  1.922760  1000.0

In [6]:
norm_meteo_df = normalize_dataframe(ds, features=["H", "T"])

In [7]:
norm_meteo_df.head(5)

,H,T
0,0.347682,-0.271171
1,0.347682,-0.271171
2,0.347682,-0.271171
3,0.345259,-0.263639
4,0.115625,-0.190686


In [8]:
meteo_array = meteo_df.to_numpy()
norm_meteo_array = norm_meteo_df.to_numpy()

## Исследование кластеризаторов

In [9]:
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV

from src.evaluation import stratified_sample_by_cluster, calculate_metrics
from src.visualization import plot_cluster_map, plot_cluster_map_period, plot_parameter_change, plot_seasonal_similarity
from src.evaluation import silhouette_scoring, random_sample

In [10]:
norm_meteo_array_sample = random_sample(norm_meteo_array, 5000)

**Замечание**: за основную метрику выбран коэффициент силуэта

Формула:

Для каждого объекта **i** вычисляется коэффициент силуэта:

$$
s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}
$$

a(i) — среднее расстояние от объекта i до всех других объектов в своём кластере (компактность)

b(i) — среднее расстояние от объекта i до всех объектов в ближайшем соседнем кластере (разделимость)

$$
\text{Silhouette Score} = \frac{1}{n} \sum_{i=1}^{n} s(i)
$$

где **n** — общее количество объектов в выборке.

Диапазон значений: от -1 до 1

| Значение | Качество кластеризации |
|----------|------------------------|
| **0.71 – 1.00** | Отличное |
| **0.51 – 0.70** | Хорошее |
| **0.26 – 0.50** | Слабое|
| **≤ 0.25** | Плохое|

### Кластеризатор KMeans

[Обзор метода кластеризации K-Means](cluster-methods/kmeans.md)

#### Поиск оптимальных гиперпараметров для кластеризатора на основе KMeans

In [11]:
from src.models.kmeans import KMeansClusterer

if not Path("models","kmeans_model.pkl").exists():

    kmeans = KMeansClusterer(random_state=42)

    param_grid_kmeans = {
        'n_clusters': [2, 3, 4, 5, 6],
        'n_init': [2, 3, 4, 5],
        'init': ['k-means++']
    }

    grid_kmeans = GridSearchCV(
        estimator=kmeans,
        param_grid=param_grid_kmeans,
        scoring=silhouette_scoring,    
        n_jobs=-1,
        verbose=1
    )    

    grid_kmeans.fit(norm_meteo_array_sample)

    kmeans_model = grid_kmeans.best_estimator_

    kmeans_model.save(Path("models") / "kmeans_model.pkl")
else:
    kmeans_model = KMeansClusterer.load(Path("models") / "kmeans_model.pkl")    
    

print("Best parameters:", kmeans_model.get_params())
print("Best score:", silhouette_scoring(kmeans_model, norm_meteo_array_sample))

labels = kmeans_model.labels_
print("Количество кластеров:", len(np.unique(labels)))

Best parameters: {'algorithm': 'lloyd', 'copy_x': True, 'init': 'k-means++', 'max_iter': 300, 'n_clusters': 2, 'n_init': 2, 'random_state': 42, 'tol': 0.0001, 'verbose': 0}
Best score: 0.3682606518268585
Количество кластеров: 2


### Gaussian Mixture Model

[Обзор метода кластеризации GMM](cluster-methods/gmm.md)

#### Поиск оптимальных гиперпараметров для кластеризатора на основе Gaussian Mixture Model

In [12]:
from src.models import GMMClusterer

if not Path("models","gmm_model.pkl").exists():

    gmm = GMMClusterer(random_state=42)

    param_grid_gmm = {
        'n_components': [2, 3, 4, 5, 6],
        'covariance_type': ['full', 'tied', 'diag', 'spherical'], 
        'n_init': [1, 2, 3, 4]    
    }

    grid_gmm = GridSearchCV(
        estimator=gmm,
        param_grid=param_grid_gmm,
        scoring=silhouette_scoring,
        cv=3,
        n_jobs=-1,
        verbose=1
    )

    grid_gmm.fit(norm_meteo_array_sample)

    gmm_model = grid_gmm.best_estimator_

    gmm_model.save(Path("models") / "gmm_model.pkl")
else:
    gmm_model = GMMClusterer.load(Path("models") / "gmm_model.pkl")

print("Best parameters:", gmm_model.get_params())
print("Best score:", silhouette_scoring(gmm_model, norm_meteo_array_sample))

labels = gmm_model.n_components
print("Количество кластеров:", len(np.unique(labels)))

Best parameters: {'covariance_type': 'spherical', 'init_params': 'kmeans', 'max_iter': 100, 'means_init': None, 'n_components': 2, 'n_init': 2, 'precisions_init': None, 'random_state': 42, 'reg_covar': 1e-06, 'tol': 0.001, 'verbose': 0, 'verbose_interval': 10, 'warm_start': False, 'weights_init': None}
Best score: 0.37550610303878784
Количество кластеров: 1


### DBSCAN model

[Обзор метода кластеризации DBSCAN](cluster-methods/dbscan.md)

In [28]:
from sklearn.model_selection import RandomizedSearchCV
from src.models import DBSCANClusterer
from scipy.stats import uniform, randint

if not Path("models","dbscan_model.pkl").exists():

    dbscan = DBSCANClusterer(random_state=42)

    param_distributions_dbscan = {
        'eps': uniform(0.6, 2.0),          
        'min_samples': randint(3, 101) 
    }
   
    grid_dbscan = RandomizedSearchCV(
        estimator=dbscan,
        param_distributions=param_distributions_dbscan,
        scoring=silhouette_scoring,
        cv=3,
        n_jobs=-1,
        verbose=1,
        n_iter=50,          
        random_state=42,   
        return_train_score=False
    )

    grid_dbscan.fit(norm_meteo_array_sample)

    dbscan_model = grid_dbscan.best_estimator_

    dbscan_model.save(Path("models") / "dbscan_model.pkl")
else:
    dbscan_model = DBSCANClusterer.load(Path("models") / "dbscan_model.pkl")

print("Best parameters:", dbscan_model.get_params())
print("Best score:", silhouette_scoring(dbscan_model, norm_meteo_array_sample))

labels = dbscan_model.labels_
print("Количество кластеров:", len(np.unique(labels)))

Best parameters: {'algorithm': 'auto', 'eps': np.float64(1.399721943430511), 'leaf_size': 30, 'metric': 'euclidean', 'metric_params': None, 'min_samples': 94, 'n_jobs': None, 'p': None}
Best score: 0.6613047122955322
Количество кластеров: 2
